In [0]:
# Databricks notebook source

# COMMAND ----------
# MAGIC %md
# MAGIC # 🥇 GOLD — Tabelas Analíticas F1
# MAGIC
# MAGIC Cria 4 tabelas otimizadas para dashboards e relatórios:
# MAGIC
# MAGIC | Tabela | Descrição |
# MAGIC |--------|-----------|
# MAGIC | `gold_classificacao_pilotos`    | Campeonato de pilotos por temporada |
# MAGIC | `gold_classificacao_construtores` | Campeonato de construtores por temporada |
# MAGIC | `gold_corridas_resumo`          | Resultado de cada corrida (vencedor, pole, etc.) |
# MAGIC | `gold_evolucao_pontos`          | Pontuação acumulada rodada a rodada por piloto |

# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.window import Window

DATABASE = "f1_db"
SILVER   = f"{DATABASE}.silver_resultados_f1"

spark.sql(f"USE {DATABASE}")

# Lê apenas corridas (exclui sprints do campeonato principal)
df_corridas = spark.table(SILVER).filter(F.col("tipo") == "corrida")
df_todos    = spark.table(SILVER)   # inclui sprints para pontos totais

print(f"📥 Corridas: {df_corridas.count()} | Com sprints: {df_todos.count()}")

# COMMAND ----------
# MAGIC %md
# MAGIC ## 1. Classificação de Pilotos

# COMMAND ----------

w_rank_pilotos = Window.partitionBy("ano").orderBy(F.desc("total_pontos"))

df_pilotos = (
    df_todos   # usa pontos de corrida + sprint
    .groupBy("ano", "piloto", "codigo_piloto", "construtor")
    .agg(
        F.sum("pontos").alias("total_pontos"),
        F.sum("pontos_sem_sprint").alias("pontos_corridas"),
        F.count("*").alias("corridas_disputadas"),
        F.sum(F.when(F.col("venceu") & (F.col("tipo") == "corrida"), 1).otherwise(0)).alias("vitorias"),
        F.sum(F.when(F.col("podio") & (F.col("tipo") == "corrida"), 1).otherwise(0)).alias("podios"),
        F.sum(F.when(F.col("pole_position"), 1).otherwise(0)).alias("poles"),
        F.sum(F.when(F.col("terminou"), 1).otherwise(0)).alias("corridas_concluidas"),
        F.sum(F.when(F.col("venceu") & (F.col("tipo") == "sprint"), 1).otherwise(0)).alias("vitorias_sprint"),
    )
    .withColumn("posicao_campeonato", F.rank().over(w_rank_pilotos))
    .withColumn("pct_conclusao",
        F.round(F.col("corridas_concluidas") / F.col("corridas_disputadas") * 100, 1)
    )
    .orderBy("ano", "posicao_campeonato")
)

display(df_pilotos)

(df_pilotos.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DATABASE}.gold_classificacao_pilotos"))
print("✅ gold_classificacao_pilotos")

# COMMAND ----------
# MAGIC %md
# MAGIC ## 2. Classificação de Construtores

# COMMAND ----------

w_rank_const = Window.partitionBy("ano").orderBy(F.desc("total_pontos"))

df_construtores = (
    df_todos
    .groupBy("ano", "construtor")
    .agg(
        F.sum("pontos").alias("total_pontos"),
        F.countDistinct("piloto").alias("num_pilotos"),
        F.count("*").alias("corridas_disputadas"),
        F.sum(F.when(F.col("venceu") & (F.col("tipo") == "corrida"), 1).otherwise(0)).alias("vitorias"),
        F.sum(F.when(F.col("podio") & (F.col("tipo") == "corrida"), 1).otherwise(0)).alias("podios"),
        F.sum(F.when(F.col("pole_position"), 1).otherwise(0)).alias("poles"),
        F.sum(F.when(F.col("terminou"), 1).otherwise(0)).alias("corridas_concluidas"),
    )
    .withColumn("posicao_campeonato", F.rank().over(w_rank_const))
    .orderBy("ano", "posicao_campeonato")
)

display(df_construtores)

(df_construtores.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DATABASE}.gold_classificacao_construtores"))
print("✅ gold_classificacao_construtores")

# COMMAND ----------
# MAGIC %md
# MAGIC ## 3. Resumo por Corrida

# COMMAND ----------

df_resumo = (
    df_corridas
    .groupBy("ano", "rodada", "corrida", "circuito", "pais", "data")
    .agg(
        F.first(F.when(F.col("posicao") == 1, F.col("piloto"))).alias("vencedor"),
        F.first(F.when(F.col("posicao") == 1, F.col("construtor"))).alias("construtor_vencedor"),
        F.first(F.when(F.col("posicao") == 2, F.col("piloto"))).alias("segundo_lugar"),
        F.first(F.when(F.col("posicao") == 3, F.col("piloto"))).alias("terceiro_lugar"),
        F.first(F.when(F.col("grid") == 1, F.col("piloto"))).alias("pole_sitter"),
        F.count("*").alias("pilotos_largaram"),
        F.sum(F.when(F.col("terminou"), 1).otherwise(0)).alias("pilotos_terminaram"),
        F.max("voltas").alias("total_voltas"),
        # Conquista da pole (converteu em vitória?)
        F.first(F.when(F.col("grid") == 1, F.col("posicao") == 1)).alias("pole_virou_vitoria"),
    )
    .withColumn("pct_conclusao",
        F.round(F.col("pilotos_terminaram") / F.col("pilotos_largaram") * 100, 1)
    )
    .orderBy("ano", "rodada")
)

display(df_resumo)

(df_resumo.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DATABASE}.gold_corridas_resumo"))
print("✅ gold_corridas_resumo")

# COMMAND ----------
# MAGIC %md
# MAGIC ## 4. Evolução de Pontos — Acumulado por Piloto/Rodada

# COMMAND ----------

# Janela de soma cumulativa por piloto dentro de cada ano
w_cumsum = (
    Window
    .partitionBy("ano", "piloto")
    .orderBy("rodada")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

# Janela de ranking por rodada (quem está liderando após cada corrida?)
w_rank_rodada = Window.partitionBy("ano", "rodada").orderBy(F.desc("pontos_acumulados"))

df_evolucao = (
    df_todos
    .groupBy("ano", "rodada", "corrida", "data", "piloto", "codigo_piloto", "construtor")
    .agg(F.sum("pontos").alias("pontos_rodada"))
    .withColumn("pontos_acumulados", F.sum("pontos_rodada").over(w_cumsum))
    .withColumn("posicao_apos_rodada", F.rank().over(w_rank_rodada))
    .withColumn("liderando", F.col("posicao_apos_rodada") == 1)
    .orderBy("ano", "piloto", "rodada")
)

display(df_evolucao.filter(F.col("posicao_apos_rodada") <= 5).limit(50))

(df_evolucao.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DATABASE}.gold_evolucao_pontos"))
print("✅ gold_evolucao_pontos")

# COMMAND ----------
# MAGIC %md
# MAGIC ## ✅ Resumo das tabelas Gold criadas

# COMMAND ----------

spark.sql(f"SHOW TABLES IN {DATABASE} LIKE 'gold*'").show(truncate=False)

📥 Corridas: 633 | Com sprints: 819


ano,piloto,codigo_piloto,construtor,total_pontos,pontos_corridas,corridas_disputadas,vitorias,podios,poles,corridas_concluidas,vitorias_sprint,posicao_campeonato,pct_conclusao
2025,Lando Norris,NOR,McLaren,423.0,394.0,30,7,18,8,26,2,1,86.7
2025,Max Verstappen,VER,Red Bull,421.0,389.0,30,8,15,9,29,2,2,96.7
2025,Oscar Piastri,PIA,McLaren,410.0,381.0,30,7,16,8,26,1,3,86.7
2025,George Russell,RUS,Mercedes,319.0,289.0,30,2,9,2,29,0,4,96.7
2025,Charles Leclerc,LEC,Ferrari,242.0,225.0,30,0,7,1,26,0,5,86.7
2025,Lewis Hamilton,HAM,Ferrari,156.0,135.0,30,0,0,1,26,1,6,86.7
2025,Andrea Kimi Antonelli,ANT,Mercedes,150.0,135.0,30,0,3,1,24,0,7,80.0
2025,Alexander Albon,ALB,Williams,73.0,70.0,30,0,0,0,22,0,8,73.3
2025,Carlos Sainz,SAI,Williams,64.0,54.0,30,0,2,0,20,0,9,66.7
2025,Fernando Alonso,ALO,Aston Martin,56.0,51.0,30,0,0,0,21,0,10,70.0


✅ gold_classificacao_pilotos


ano,construtor,total_pontos,num_pilotos,corridas_disputadas,vitorias,podios,poles,corridas_concluidas,posicao_campeonato
2025,McLaren,833.0,2,60,14,34,16,52,1
2025,Mercedes,469.0,2,60,2,12,3,53,2
2025,Red Bull,451.0,3,60,8,15,9,51,3
2025,Ferrari,398.0,2,60,0,7,2,52,4
2025,Williams,137.0,2,60,0,2,0,42,5
2025,RB F1 Team,92.0,3,60,0,1,0,40,6
2025,Aston Martin,89.0,2,59,0,0,0,38,7
2025,Haas F1 Team,79.0,2,60,0,0,0,45,8
2025,Sauber,70.0,2,60,0,1,0,36,9
2025,Alpine F1 Team,22.0,3,60,0,0,0,29,10


✅ gold_classificacao_construtores


ano,rodada,corrida,circuito,pais,data,vencedor,construtor_vencedor,segundo_lugar,terceiro_lugar,pole_sitter,pilotos_largaram,pilotos_terminaram,total_voltas,pole_virou_vitoria,pct_conclusao
2025,1,Australian Grand Prix,Albert Park Grand Prix Circuit,Australia,2025-03-16,Lando Norris,McLaren,null,null,Lando Norris,20,14,57,true,70.0
2025,2,Chinese Grand Prix,Shanghai International Circuit,China,2025-03-23,Oscar Piastri,McLaren,null,null,Oscar Piastri,20,13,56,true,65.0
2025,3,Japanese Grand Prix,Suzuka Circuit,Japan,2025-04-06,Max Verstappen,Red Bull,null,null,Max Verstappen,20,19,53,true,95.0
2025,4,Bahrain Grand Prix,Bahrain International Circuit,Bahrain,2025-04-13,Oscar Piastri,McLaren,null,null,Oscar Piastri,20,18,57,true,90.0
2025,5,Saudi Arabian Grand Prix,Jeddah Corniche Circuit,Saudi Arabia,2025-04-20,Oscar Piastri,McLaren,null,null,null,20,14,50,null,70.0
2025,6,Miami Grand Prix,Miami International Autodrome,USA,2025-05-04,Oscar Piastri,McLaren,null,null,null,20,13,57,null,65.0
2025,7,Emilia Romagna Grand Prix,Autodromo Enzo e Dino Ferrari,Italy,2025-05-18,Max Verstappen,Red Bull,null,null,null,20,18,63,null,90.0
2025,8,Monaco Grand Prix,Circuit de Monaco,Monaco,2025-05-25,Lando Norris,McLaren,null,null,Lando Norris,20,5,78,true,25.0
2025,9,Spanish Grand Prix,Circuit de Barcelona-Catalunya,Spain,2025-06-01,Oscar Piastri,McLaren,null,null,Oscar Piastri,19,17,66,true,89.5
2025,10,Canadian Grand Prix,Circuit Gilles Villeneuve,Canada,2025-06-15,George Russell,Mercedes,null,null,George Russell,20,8,70,true,40.0


✅ gold_corridas_resumo


ano,rodada,corrida,data,piloto,codigo_piloto,construtor,pontos_rodada,pontos_acumulados,posicao_apos_rodada,liderando
2025,1,Australian Grand Prix,2025-03-16,Alexander Albon,ALB,Williams,10.0,10.0,5,false
2025,1,Australian Grand Prix,2025-03-16,Andrea Kimi Antonelli,ANT,Mercedes,12.0,12.0,4,false
2025,2,Chinese Grand Prix,2025-03-23,Andrea Kimi Antonelli,ANT,Mercedes,10.0,22.0,5,false
2025,3,Japanese Grand Prix,2025-04-06,Andrea Kimi Antonelli,ANT,Mercedes,8.0,30.0,5,false
2025,4,Bahrain Grand Prix,2025-04-13,Charles Leclerc,LEC,Ferrari,12.0,32.0,5,false
2025,5,Saudi Arabian Grand Prix,2025-04-20,Charles Leclerc,LEC,Ferrari,15.0,47.0,5,false
2025,6,Miami Grand Prix,2025-05-04,Charles Leclerc,LEC,Ferrari,6.0,53.0,5,false
2025,7,Emilia Romagna Grand Prix,2025-05-18,Charles Leclerc,LEC,Ferrari,8.0,61.0,5,false
2025,8,Monaco Grand Prix,2025-05-25,Charles Leclerc,LEC,Ferrari,18.0,79.0,5,false
2025,9,Spanish Grand Prix,2025-06-01,Charles Leclerc,LEC,Ferrari,15.0,94.0,5,false


✅ gold_evolucao_pontos
+--------+-------------------------------+-----------+
|database|tableName                      |isTemporary|
+--------+-------------------------------+-----------+
|f1_db   |gold_classificacao_construtores|false      |
|f1_db   |gold_classificacao_pilotos     |false      |
|f1_db   |gold_corridas_resumo           |false      |
|f1_db   |gold_evolucao_pontos           |false      |
+--------+-------------------------------+-----------+

